In [2]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

raw_data = pd.read_csv('../data/raw/housing/housing.csv')

print(raw_data.head())

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  


# Problemas:
## 1. Problemas de Precision:

- Datos Atipicos (Outliers)
- Inconsistencia en escalas
- Valores Duplicdos/Repetidos (Pablo Punto Pedir - Mail Erick Preguntar)
- Inconsistencias en los datos

## 2. Problema de Completitud:

- Valores Ausentes: Nulos

###  Todo esto debe hacerse antes del muestreo. Ya que sino llega la data contaminada. Se resuelven problemas de calidad antes de muestrear.



In [3]:
# Completitud: Valores Ausentes.

raw_data.isna().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [4]:
raw_data[raw_data['total_bedrooms'].isna()]

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
182,-118.27,34.04,13.0,1784.0,NaN,2158.0,682.0,1.7038,118100.0,<1H OCEAN
327,-117.65,34.04,15.0,3393.0,NaN,2039.0,611.0,3.9336,151000.0,INLAND
366,-122.50,37.75,44.0,1819.0,NaN,1137.0,354.0,3.4919,271800.0,NEAR OCEAN
477,-117.99,34.14,30.0,2346.0,NaN,1988.0,474.0,2.5625,153000.0,INLAND
495,-114.59,34.83,41.0,812.0,NaN,375.0,158.0,1.7083,48500.0,INLAND
...,...,...,...,...,...,...,...,...,...,...
19880,-118.23,33.94,36.0,1110.0,NaN,1417.0,302.0,2.3333,92100.0,<1H OCEAN
19952,-119.19,34.20,18.0,3620.0,NaN,3171.0,779.0,3.3409,220500.0,NEAR OCEAN
20088,-119.73,36.83,8.0,3602.0,NaN,1959.0,580.0,5.3478,138800.0,INLAND
20325,-118.88,34.17,15.0,4260.0,NaN,1701.0,669.0,5.1033,410700.0,<1H OCEAN


In [5]:
# Hacer un EDA de los datos ausentes para entender si hay alguna relación entre ellos y otras variables.


In [6]:
# Hay tres formas de lidiar con los datos ausentes:
# 1. Eliminar las filas con datos ausentes. 
# 2. Imputar los datos ausentes con la media, mediana o moda. O algoritmo. Si se rellena con una metrica de tendencia central se dañan los datos, es mejor usar otro algoritmo. Regrsion lineal, knn, etc.
# 3. Borrar la columna con datos ausentes si tiene muchos datos ausentes.

# Pandas esta constituido por series, es un nd array de numpy. Si se elimina una fila, se elimina un elemento de cada serie. Si se elimina una columna, se elimina una serie completa.
# No se le puede agregar una columna a un dataframe, se le puede agregar una serie. Si se elimina una fila, se elimina un elemento de cada serie. Si se elimina una columna, se elimina una serie completa.

raw_data_na = raw_data.isna().sum().reset_index()
raw_data_na.columns = ['column_name', 'na_count']
raw_data_na['na_percentage'] = raw_data_na['na_count'] / len(raw_data) * 100
raw_data_na

,column_name,na_count,na_percentage
0,longitude,0,0.000000
1,latitude,0,0.000000
2,housing_median_age,0,0.000000
3,total_rooms,0,0.000000
4,total_bedrooms,207,1.002907
5,population,0,0.000000
6,households,0,0.000000
7,median_income,0,0.000000
8,median_house_value,0,0.000000
9,ocean_proximity,0,0.000000


In [7]:
# Funciones de pandas siemper regresan el resultado no guardan el resultado en el dataframe original, a menos que se use el parametro inplace=True. Es mejor no usar inplace=True porque puede causar problemas de memoria y es mas facil de entender el codigo sin el.
clean_data = raw_data.dropna(subset=['total_bedrooms']).reset_index()
# clean_data = raw_data.fillna(raw_data['total_bedrooms'].median())
# clean_data = raw_data.drop(columns=['total_bedrooms'])
clean_data.isna().sum()
clean_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20433 entries, 0 to 20432
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   index               20433 non-null  int64  
 1   longitude           20433 non-null  float64
 2   latitude            20433 non-null  float64
 3   housing_median_age  20433 non-null  float64
 4   total_rooms         20433 non-null  float64
 5   total_bedrooms      20433 non-null  float64
 6   population          20433 non-null  float64
 7   households          20433 non-null  float64
 8   median_income       20433 non-null  float64
 9   median_house_value  20433 non-null  float64
 10  ocean_proximity     20433 non-null  object 
dtypes: float64(9), int64(1), object(1)
memory usage: 1.7+ MB


### Tipos de Clase en SKLEARN:
1. Estimador
2. Transformador
3. Predictor

In [8]:
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
# mediana no puede usarse en variables de texto.

imputer = SimpleImputer(strategy='median')

# seleccionar filas numericas

data_pre_imputed = raw_data.select_dtypes(include=[np.number])
imputer.fit(data_pre_imputed)
clean_data_imputed = imputer.transform(data_pre_imputed) # devuelve un array de numpy, hay que convertirlo a dataframe de pandas
imputer.feature_names_in_
clean_data_imputed = pd.DataFrame(clean_data_imputed, columns=data_pre_imputed.columns, index=data_pre_imputed.index) # type: ignore
clean_data_imputed

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0
...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0


In [9]:
imputer.strategy # type: ignore

'median'

In [10]:
# Problema outliers:
# 1. IQR (Interquartile Range): Se calcula el rango intercuartílico (IQR) y se eliminan los valores que están por debajo de Q1 - 1.5*IQR o por encima de Q3 + 1.5*IQR.
# 2. Z-score: Se calcula el z-score para cada valor y se eliminan los valores que tienen un z-score mayor a un umbral (por ejemplo, 3). Estar dentro de 3 desciaciones
# 3. Metodo de ensamblaje: Se pueden usar algoritmos de ensamble como Random Forest o Isolation Forest para detectar y eliminar outliers. COnjunto de modelos mas pqeueños. Ensamblamos modelos mas pequeños

# Filtrar Outliers por 3 veses desviacion estandar o por el IQR.

# Metodo Ensamblaje:
# identificar los atipicos con un predictor inteligente, existen:
from sklearn.ensemble import IsolationForest
isolation_forest = IsolationForest(contamination=0.01) # se asume que el 1% de los datos son outliers

In [15]:
# Manejo de variables de texto y categoricas

datos_categoricos = clean_data.select_dtypes(include=['object'])

datos_cat = clean_data[['ocean_proximity']]
datos_cat['ocean_proximity'].value_counts()

# Nominales: Sin ORden
# Ordinales: Con ORden ocean proximity es ordinal ya que mas cerca 

ocean_proximity
<1H OCEAN     9034
INLAND        6496
NEAR OCEAN    2628
NEAR BAY      2270
ISLAND           5
Name: count, dtype: int64

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

codificador_ordinal = OrdinalEncoder()
datos_cat_ordinal = codificador_ordinal.fit_transform(datos_cat)
# datos_cat_ordinal = pd.DataFrame(datos_cat_ordinal, columns=datos_cat.columns, index=datos_cat.index) # type: ignore
datos_cat_ordinal

array([[3.],
       [3.],
       [3.],
       ...,
       [1.],
       [1.],
       [1.]])

In [17]:
codificador_ordinal.categories_ # type: ignore

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [19]:
# Estas es una doficiacion ordinal y para la nominal se usa OneHotEncoder, que crea una columna para cada categoria y asigna un 1 o un 0 dependiendo de si la fila pertenece a esa categoria o no. Es importante no usar el ordinal encoder para variables nominales porque se le esta dando un orden a las categorias que no existe.

pd.get_dummies(clean_data, drop_first=True) # Cada posibidliad de las columnas categoricas las puso false true.

# se usa el drop first true para evitar reredundancias yaque si todas son falsas, entonces la categoria es la que se elimino. Si no se usa el drop first true, entonces se puede caer en la trampa de la multicolinealidad, que es cuando una variable es una combinacion lineal de otras variables. Esto puede causar problemas en algunos modelos como la regresion lineal.

# que debemos hacer con esto one hot encoder o ordinal encoder? depende del modelo que se va a usar, si el modelo es un arbol de decision, no importa el orden de las categorias, entonces se puede usar one hot encoder. Si el modelo es una regresion lineal, el orden de las categorias si importa, entonces se debe usar ordinal encoder.



# clean_data_final = pd.concat([clean_data_imputed, datos_cat_ordinal], axis=1)


,index,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,False,False,True,False
1,1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,False,False,True,False
2,2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,False,False,True,False
3,3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,False,False,True,False
4,4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20428,20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,True,False,False,False
20429,20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,True,False,False,False
20430,20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,True,False,False,False
20431,20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,True,False,False,False


In [ ]:
clean_data['ocean_proximity_ordinal'] = datos_cat_ordinal.astype(int) # type: ignore

clean_data.head()


,index,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,ocean_proximity_ordinal
0,0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,3
1,1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,3
2,2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,3
3,3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,3
4,4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,3


In [ ]:
clean_data.drop(columns=['index'], inplace=True) # type: ignore
clean_data.drop(columns=['ocean_proximity'], inplace=True) # type: ignore

In [27]:
# Precision, completitud, vlores ausentes, imputacion, valor tendencia central, outliers, herramientas machine learning para imputar.
# Para detectar poutliers usamos el metodo de ensamblaje, que es un metodo mas inteligente que los metodos tradicionales como el IQR o el Z-score, ya que no se basa en un umbral fijo, sino que se adapta a los datos.
# Vimos temas de variables categoricas, como el ordinal encoder y el one hot encoder, y cuando usar cada uno. El ordinal encoder se usa para variables ordinales, mientras que el one hot encoder se usa para variables nominales. Es importante no usar el ordinal encoder para variables nominales porque se le esta dando un orden a las categorias que no existe.
# Tenemos dos tipos, nominal y ordinal.

# Con esto ya tenemos un dataset limpio y listo para ser usado en un modelo de machine learning. Es importante recordar que el proceso de limpieza de datos es un proceso iterativo, es decir, que se debe revisar el dataset varias veces para asegurarse de que no hay errores o problemas que puedan afectar el rendimiento del modelo.

# Falta revisar la escala. Algoritmos ML tienen favoritismos a valroes de escalas grandes.

from sklearn.preprocessing import MinMaxScaler, StandardScaler
# Hay dos tipos de escalamiento
# 1. MinMaxScaler: Escala los datos a un rango entre 0 y 1. Es sensible a los outliers. Default es de -1 a 1.
# 2. StandardScaler: Escala los datos para que tengan media 0 y desviacion estandar 1. No es sensible a los outliers.

# PUNTO PABLO ALVARADO DE ESTO

# Feature Scaling: 
# 1. Normalizacion: Se escala cada caracteristica a un rango entre 0 y 1. Es sensible a los outliers. Esto es el MinMaxScaler de sklearn.
# 2. Estandarizacion: Se escala cada caracteristica para que tenga media 0 y desviacion estandar 1. No es sensible a los outliers. Esto es el StandardScaler de sklearn.


# Son trasnformadores tienen fit i y transofrm

min_max_scaler = MinMaxScaler(feature_range=(-1, 1))
standard_scaler = StandardScaler()


clean_data_scaled_min_max = min_max_scaler.fit_transform(clean_data)
clean_data_scaled_min_max


array([[-0.57768924,  0.13496281,  0.56862745, ...,  0.07933684,
         0.80453276,  0.5       ],
       [-0.57569721,  0.13071201, -0.21568627, ...,  0.07605412,
         0.41649313,  0.5       ],
       [-0.57968127,  0.12858661,  1.        , ..., -0.06794389,
         0.39010148,  0.5       ],
       ...,
       [-0.37649402,  0.46439957, -0.37254902, ..., -0.83447125,
        -0.6812343 , -0.5       ],
       [-0.39641434,  0.46439957, -0.33333333, ..., -0.8114095 ,
        -0.71257438, -0.5       ],
       [-0.38047809,  0.45164718, -0.41176471, ..., -0.73949325,
        -0.69319302, -0.5       ]])